# 02 - Feature Engineering

This notebook turns the cleaned match data from `01` into pre-match features: how each team was performing going into every game. To prevent data leakage, every feature is built only from matches played *before* the one it describes, so the model never sees information it wouldn't have at kickoff.

**Input:** `data/processed/matches.csv`  
**Output:** `data/processed/features.csv`

## Imports, settings, and load

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_PATH = Path("../data/processed/matches.csv")
ROLLING_WINDOW = 5
feature_cols = []

matches = pd.read_csv(PROCESSED_PATH, parse_dates=["Date"])

assert matches.shape[0] == 4560, "There should be 4560 matches across 12 seasons"
assert pd.api.types.is_datetime64_any_dtype(matches["Date"]), "Date column should be datetime type"


## Points per Match

Each result is converted into league points for both teams (win = 3, draw = 1, loss = 0). Points form is one of the core features built later, and this gives every match a numeric outcome per team that can be averaged over recent games.

In [3]:
home_points = {"H": 3, "D": 1, "A": 0}
away_points = {"H": 0, "D": 1, "A": 3}

matches["HomePts"] = matches["FTR"].map(home_points)
matches["AwayPts"] = matches["FTR"].map(away_points)

assert (matches["HomePts"] + matches["AwayPts"] == 3 - (matches["FTR"] == "D")).all(), "Each match should total 3 points, or 2 for a draw"

## Home-Perspective Table

To track each team's history, every match is split into two rows, one from each team's point of view. This first table describes each match from the home team's side, renaming its columns to neutral names (`goals_for`, `goals_against`, etc.) so home and away rows can later be stacked into a single table.

In [6]:
home_view = matches[["Season", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "HS", "AS", "HST", "AST", "HC", "AC", "HY", "AY", "HR", "AR", "HomePts"]].copy()
home_view = home_view.rename(columns = {"HomeTeam": "team", "AwayTeam": "opponent", "FTHG": "goals_for", "FTAG": "goals_against", "HS": "shots_for", "AS": "shots_against", "HST": "sot_for", "AST": "sot_against" , "HC": "corners_for", "AC": "corners_against", "HY": "yellows_for", "AY": "yellows_against", "HR": "reds_for", "AR": "reds_against", "HomePts": "points"})
home_view["is_home"] = 1
assert home_view.shape[0] == 4560, "Should be 4560 matches in 12 seasons"

## Away-Perspective Table

The same matches, now described from the away team's side. Every `_for` and `_against` pair is flipped, so the away team's stats become `_for` and the home team's become `_against`. After stacking, a column like `sot_for` always means "this team's shots on target," regardless of venue.

In [8]:
away_view = matches[["Season", "Date", "AwayTeam", "HomeTeam", "FTAG", "FTHG", "AS", "HS", "AST", "HST", "AC", "HC", "AY", "HY", "AR", "HR", "AwayPts"]].copy()
away_view = away_view.rename(columns = {"AwayTeam": "team", "HomeTeam": "opponent", "FTAG": "goals_for", "FTHG": "goals_against", "AS": "shots_for", "HS": "shots_against", "AST": "sot_for", "HST": "sot_against", "AC": "corners_for", "HC": "corners_against", "AY": "yellows_for", "HY": "yellows_against", "AR": "reds_for", "HR": "reds_against", "AwayPts": "points"})
away_view["is_home"] = 0
assert away_view.shape[0] == 4560, "Should be 4560 matches in 12 seasons"
assert list(away_view.columns) == list(home_view.columns), "Home and away view columns should match"